# MovieLens 25M — Phase 2: Standalone Models & Validation Benchmarks

**Goal.** Implement and fairly compare four standalone recommenders under the frozen Phase 1 protocol, then record validation benchmarks, tuning evidence and resource requirements.

| Task | What it covers |
|---|---|
| P2.1 | Popularity baseline (ranking) |
| P2.2 | Regularized bias baseline (rating) |
| P2.3 | Item-based CF (rating) |
| P2.4 | Matrix factorization / ALS (rating) |
| P2.5 | Comparable tuning budgets and selection |
| P2.6 | Common recommendation lists (shared eligibility and ranking) |
| P2.7 | Standalone metrics (NDCG@10, Recall@10, coverage, exposure, RMSE, MAE) |
| P2.8 | Runtime and memory requirements |
| P2.9 | Peer review / independent reproduction |

Each step below explains the idea and the formula, then runs the real code in `src/` so results are verifiable end to end.

## 0. How to use this notebook

- Run `python -m src.prepare_datasets` once so `data/processed/` exists.
- Select the project `.venv` kernel (needs numpy, pandas, scipy, psutil, scikit-learn).
- Run cells top to bottom; later cells reuse `history`, `catalogue`, `positives`, `cohort`, `selection`, `fallback`.
- Model settings come from `outputs/phase2_selected_config_extended.json`, the same file `src.run_phase2_validation` reads, so the notebook and the runner evaluate identical configurations.
- Item-CF and MF fits take about four minutes each.

In [1]:
import os
import sys
import platform
from pathlib import Path
from time import perf_counter

import numpy as np
import pandas as pd
import psutil

# Locate the project root (the folder that contains src/) and run from there.
root = Path.cwd()
while root != root.parent and not (root / 'src').is_dir():
    root = root.parent
os.chdir(root)
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

PROCESS = psutil.Process()

def rss_mb():
    """Resident set size of this process in MB."""
    return PROCESS.memory_info().rss / (1024 ** 2)

pd.set_option('display.width', 160)
print('Project root :', root)
print('Python       :', platform.python_version())
print('NumPy/Pandas :', np.__version__, '/', pd.__version__)

Project root : c:\Users\Truong\Desktop\projects\statistical-learning-movielens
Python       : 3.13.2
NumPy/Pandas : 2.5.4 / 3.1.1


## 1. Evaluation protocol (frozen in Phase 1)

All models share one protocol.

- **History** $H$: every rating strictly before `2017-01-01` UTC ($|H| = 20{,}798{,}765$).
- **Outcomes** $O$: 2017 ratings, used only for scoring, never for fitting. No 2018–2019 data is used.
- **Catalogue** $\mathcal{C}$: 17,965 movies with at least 10 pre-2017 ratings.
- **Cohort** $\mathcal{U}$: 4,778 users with at least one eligible 2017 positive (169,368 positives).
- **Positive**: an observed rating strictly greater than 3.
- **Candidates** for user $u$: $\mathcal{C}$ minus the movies $u$ rated in $H$.
- **Ranking**: unclipped scores, descending; exact ties broken by ascending `movieId`; top 10.
- **Fallback**: a separately fitted bias model ($\lambda_u = \lambda_i = 10$, 20 passes) fills missing scores and missing rating predictions. Tuning and the runner use the same fallback (`src.tune_models.FALLBACK_SETTINGS`); it is distinct from the tuned bias baseline.

Eligibility uses only pre-cutoff information, so no future outcome leaks into training or cohort construction.

In [2]:
from src.ratings import load_ratings
from src.splits import VALIDATION_START, TEST_START
from src.pipeline import EvaluationCohort, score_grid
from src.bias import BiasModel
from src.run_phase2_validation import (
    SELECTED_CONFIG_PATH,
    load_selected_config,
    rating_predictions,
)
from src.tune_models import FALLBACK_SETTINGS

catalogue = pd.read_csv('data/processed/eligible_movies.csv')
positives = pd.read_csv('data/processed/validation_positives.csv')

ratings = load_ratings()
history = ratings.loc[
    ratings['timestamp'] < VALIDATION_START, ['userId', 'movieId', 'rating']
].copy()

# Shared rating-prediction pairs for RMSE/MAE (cohort users x catalogue movies).
outcomes_2017 = ratings.loc[
    (ratings['timestamp'] >= VALIDATION_START) & (ratings['timestamp'] < TEST_START),
    ['userId', 'movieId', 'rating'],
]
del ratings
rating_pairs = (
    outcomes_2017.loc[
        outcomes_2017['userId'].isin(positives['userId'].unique())
        & outcomes_2017['movieId'].isin(catalogue['movieId'])
    ]
    .drop_duplicates(['userId', 'movieId'])
    .reset_index(drop=True)
)

cohort = EvaluationCohort(catalogue, positives)
cohort.set_history(history)

# P2.5 selection (same file and checks as the runner) and the shared fallback.
selection = load_selected_config()
fallback = BiasModel(**FALLBACK_SETTINGS).fit(history)

print(f'History rows     : {len(history):,}')
print(f'Catalogue movies : {len(catalogue):,}')
print(f'Evaluated users  : {len(cohort.users):,}')
print(f'Positives        : {len(positives):,}')
print(f'Rating pairs     : {len(rating_pairs):,}')
print(f'Selection file   : {SELECTED_CONFIG_PATH}')
print(f'Selection        : {selection}')
print(f'Fallback         : bias {FALLBACK_SETTINGS}')

History rows     : 20,798,765
Catalogue movies : 17,965
Evaluated users  : 4,778
Positives        : 169,368
Rating pairs     : 275,061
Selection file   : outputs\phase2_selected_config_extended.json
Selection        : {'bias': {'lambda_movie': 320.0, 'lambda_user': 320.0}, 'item_cf': {'min_similarity': 0.0, 'top_k': 3}, 'matrix_factorization': {'n_factors': 24, 'regularization': 30.0}}
Fallback         : bias {'lambda_user': 10.0, 'lambda_movie': 10.0, 'max_passes': 20}


## 2. P2.1 — Popularity baseline (ranking)

**Idea.** Recommend what the crowd watched most.

**Score** (rating count in the history, independent of star value):

$$\text{pop}(i) = \sum_{(u,i) \in H} 1$$

The score is the same for every user; personalization comes only from excluding each user's own history. Popularity yields ranking scores, not star predictions, so RMSE and MAE do not apply.

In [3]:
from src.popularity import PopularityModel

def evaluate_model(model, cohort, fallback=fallback):
    """Fit, score the candidate grid (fallback-filled), rank, and summarise."""
    t0 = perf_counter()
    model = model.fit(history)
    fit_seconds = perf_counter() - t0

    t0 = perf_counter()
    scores, missing = score_grid(model, cohort, fallback=fallback)
    score_seconds = perf_counter() - t0

    ranked = cohort.rank_top_k(scores)
    summary, user_metrics = cohort.summary(ranked)
    return {
        'name': model.name,
        'model_obj': model,
        'summary': summary,
        'user_metrics': user_metrics,
        'ranked': ranked,
        'missing_scores': missing,
        'fit_seconds': fit_seconds,
        'score_seconds': score_seconds,
    }

pop_result = evaluate_model(PopularityModel(), cohort)
pd.Series(pop_result['summary'])

users                    4778.000000
mean_ndcg_at_10             0.075212
mean_recall_at_10           0.023637
coverage_movies           287.000000
coverage_fraction           0.015976
less_popular_exposure       0.000000
dtype: float64

## 3. P2.2 — Regularized bias baseline (rating)

**Model.**

$$\hat r_{ui} = \mu + b_u + b_i$$

**Objective** (regularized least squares over observed ratings):

$$\min_{b_u, b_i} \sum_{(u,i) \in H} (r_{ui} - \mu - b_u - b_i)^2 + \lambda_u \sum_u b_u^2 + \lambda_i \sum_i b_i^2$$

**Alternating updates** (closed form when the other side is fixed):

$$b_u \leftarrow \frac{\sum_{i \in I_u} (r_{ui} - \mu - b_i)}{|I_u| + \lambda_u}, \qquad b_i \leftarrow \frac{\sum_{u \in U_i} (r_{ui} - \mu - b_u)}{|U_i| + \lambda_i}$$

The terms $|I_u| + \lambda_u$ shrink noisy biases toward zero, which matters most for users and movies with few ratings. A missing user or movie gets bias 0, which is what makes the bias model the natural fallback for unsupported predictions.

**Selected setting:** $\lambda_u = \lambda_i = 320$, the largest value tried. Such strong shrinkage keeps rarely rated movies out of the top 10, which raises NDCG@10 but costs rating accuracy.

Ranking uses raw unclipped predictions; `predict` clips to the MovieLens range [0.5, 5] only for RMSE and MAE.

In [4]:
bias_result = evaluate_model(BiasModel(**selection['bias']), cohort)
pd.Series(bias_result['summary'])

users                    4778.000000
mean_ndcg_at_10             0.052504
mean_recall_at_10           0.018621
coverage_movies           154.000000
coverage_fraction           0.008572
less_popular_exposure       0.000167
dtype: float64

## 4. P2.3 — Item-based CF (rating)

**Idea.** Movies liked by the same people are similar; predict a user's rating from how they rated similar movies.

**Adjusted-cosine similarity** (subtract each user mean $\bar r_u$):

$$s_{ij} = \frac{\sum_{u \in U_{ij}} (r_{ui} - \bar r_u)(r_{uj} - \bar r_u)}{\sqrt{\sum_u (r_{ui} - \bar r_u)^2} \sqrt{\sum_u (r_{uj} - \bar r_u)^2}}$$

Keep only positive similarities, then each item keeps its top-$K$ neighbours $N_K(i)$.

**Prediction** blends the candidate item mean with similarity-weighted residuals over the candidate's **own** neighbours that the user has rated:

$$\hat r_{ui} = \bar r_i + \frac{\sum_{j \in N_K(i)} s_{ij} (r_{uj} - \bar r_j)}{\sum_{j \in N_K(i)} |s_{ij}|}$$

The original benchmark read the neighbour matrix the other way round (scoring $i$ from items that listed $i$ among *their* top $K$); the code now uses $N_K(i)$ as written above.

If the user has rated none of $N_K(i)$, the score is `NaN` and the shared fallback fills it. Using residuals keeps scores graded instead of saturating at the star ceiling, which would flatten the ranking.

**Selected setting:** $K = 3$, the smallest value tried. With so few neighbours, about 95% of the score grid comes from the fallback, so Item-CF's ranking results describe Item-CF scores mixed with fallback scores.

In [5]:
from src.item_cf import ItemCFModel

item_cf_result = evaluate_model(ItemCFModel(**selection['item_cf']), cohort)
missing = item_cf_result['missing_scores']
grid_cells = len(cohort.users) * len(cohort.movie_ids)
print(f'fallback-filled cells: {missing:,} of {grid_cells:,} ({missing / grid_cells:.2%})')
pd.Series(item_cf_result['summary'])

fallback-filled cells: 81,597,241 of 85,836,770 (95.06%)


users                    4778.000000
mean_ndcg_at_10             0.059745
mean_recall_at_10           0.025483
coverage_movies          4627.000000
coverage_fraction           0.257556
less_popular_exposure       0.159272
dtype: float64

## 5. P2.4 — Matrix factorization (explicit ALS with biases)

**Model.**

$$\hat r_{ui} = \mu + b_u + b_i + \mathbf{p}_u^\top \mathbf{q}_i$$

**Objective** over observed ratings with L2 regularization:

$$\min_{P, Q, b} \sum_{(u,i) \in H} \big(r_{ui} - \mu - b_u - b_i - \mathbf{p}_u^\top \mathbf{q}_i\big)^2 + \lambda \big(\lVert \mathbf{p}_u \rVert^2 + \lVert \mathbf{q}_i \rVert^2\big) + \lambda_u b_u^2 + \lambda_i b_i^2$$

**Alternating least squares.** With one side fixed, each row is a ridge regression with a closed form. For user $u$ with rated items $I_u$:

$$\mathbf{p}_u = \Big( \sum_{i \in I_u} \mathbf{q}_i \mathbf{q}_i^\top + \lambda I \Big)^{-1} \sum_{i \in I_u} \mathbf{q}_i (r_{ui} - \mu - b_u - b_i)$$

and symmetrically for $\mathbf{q}_i$. Each pass updates $b_u$, then $b_i$, then $P$, then $Q$ (Gauss–Seidel style). Fitting stops after 20 passes or when the training RMSE changes by less than $10^{-4}$; that training-RMSE monitor clips to [1, 5] (unchanged, so fits match tuning) and does not affect the reported RMSE/MAE.

**Regularization matters:** a weak $\lambda$ overfits. The validation-selected config is $k = 24$ factors, $\lambda = 30$, with bias regularization $\lambda_u = \lambda_i = 10$ and seed 0. The $(16, 15)$ setting is within 0.0001 NDCG@10 and covers more movies.

In [6]:
from src.matrix_factorization import MatrixFactorizationModel

mf_model = MatrixFactorizationModel(**selection['matrix_factorization'])
mf_result = evaluate_model(mf_model, cohort)
print(f"iterations: {mf_result['model_obj'].iterations_}, "
      f"training RMSE (monitor): {mf_result['model_obj'].training_rmse_:.4f}")
pd.Series(mf_result['summary'])

iterations: 19, training RMSE (monitor): 0.6922


users                    4778.000000
mean_ndcg_at_10             0.095484
mean_recall_at_10           0.040890
coverage_movies          1182.000000
coverage_fraction           0.065795
less_popular_exposure       0.272625
dtype: float64

## 6. P2.6 — Shared recommendation lists

Every model goes through the same pipeline (`src/pipeline.py`):

1. `score_candidates(users, movies)` returns a dense grid of **unclipped** scores.
2. Non-finite scores are replaced by the shared bias fallback ($\lambda = 10/10$) via `score_grid`, and the count is reported.
3. Each user's already-rated movies are set to $-\infty$.
4. Scores are negated and a **stable** top-10 argsort is taken. Because the catalogue is stored in ascending `movieId` order, stable sorting breaks exact ties by ascending `movieId` (the Phase 1 rule).

This guarantees identical eligibility, exclusion and ranking across models, which is a hard Phase 2 acceptance criterion.

In [7]:
results = {
    'popularity': pop_result,
    'bias': bias_result,
    'item_cf': item_cf_result,
    'matrix_factorization': mf_result,
}

ranking_rows = []
for key, res in results.items():
    s = res['summary']
    ranking_rows.append({
        'model': key,
        'mean_ndcg_at_10': s['mean_ndcg_at_10'],
        'mean_recall_at_10': s['mean_recall_at_10'],
        'coverage_movies': s['coverage_movies'],
        'coverage_fraction': s['coverage_fraction'],
        'less_popular_exposure': s['less_popular_exposure'],
        'missing_scores': res['missing_scores'],
    })
ranking_table = pd.DataFrame(ranking_rows).set_index('model')
ranking_table

,mean_ndcg_at_10,mean_recall_at_10,coverage_movies,coverage_fraction,less_popular_exposure,missing_scores
model,,,,,,
popularity,0.075212,0.023637,287,0.015976,0.000000,0
bias,0.052504,0.018621,154,0.008572,0.000167,0
item_cf,0.059745,0.025483,4627,0.257556,0.159272,81597241
matrix_factorization,0.095484,0.040890,1182,0.065795,0.272625,0


## 7. P2.7 — Metrics

**Ranking metrics** (per user with at least one eligible positive; binary relevance):

$$\text{DCG@10} = \sum_{k=1}^{10} \frac{\mathbb{1}[y_k \in \text{positives}]}{\log_2 (k+1)}, \qquad \text{NDCG@10} = \frac{\text{DCG@10}}{\text{IDCG@10}}$$

$$\text{Recall@10} = \frac{|\text{top-10} \cap \text{positives}|}{|\text{positives}|}$$

**Catalogue coverage** at 10 = distinct recommended titles divided by $|\mathcal{C}|$.

**Less-popular exposure** = share of returned slots outside the top fifth of movies by pre-2017 count (that top fifth holds about 89.87 percent of historical ratings).

**Rating metrics** on aligned observed pairs. Missing predictions are filled by the fallback, then every prediction is clipped to the MovieLens range, $\hat r_i \leftarrow \min(\max(\hat r_i, 0.5), 5)$ (`rating_predictions` in the runner):

$$\text{RMSE} = \sqrt{\frac{1}{n} \sum_{i=1}^{n} (r_i - \hat r_i)^2}, \qquad \text{MAE} = \frac{1}{n} \sum_{i=1}^{n} |r_i - \hat r_i|$$

In [8]:
from src.metrics import rmse, mae

actual = rating_pairs['rating'].to_numpy(dtype=np.float64)
rating_rows = []
for key, res in results.items():
    model = res['model_obj']
    try:
        # Same path as the runner: fallback-fill, then clip to [0.5, 5].
        predicted = rating_predictions(model, fallback, rating_pairs)
    except NotImplementedError:
        continue  # popularity has no star predictions
    rating_rows.append({
        'model': key,
        'rmse': rmse(actual, predicted),
        'mae': mae(actual, predicted),
    })
rating_table = pd.DataFrame(rating_rows).set_index('model')
rating_table

,rmse,mae
model,,
bias,0.876850,0.659556
item_cf,0.879636,0.649669
matrix_factorization,0.799053,0.590185


## 8. P2.8 — Resource requirements

Record fit time, score time and resident memory per model, with the hardware stated so numbers are comparable.

$$\text{RSS (MB)} = \frac{\text{resident bytes}}{1024^2}$$

The cell below reports the current process RSS after all runs. The full runner records one RSS reading after each model's scoring in `outputs/phase2_validation_summary.csv`, in a process shared by all models. Neither is a per-model peak. Timings depend on the machine, so compare them only within one run.

In [9]:
resource_rows = []
for key, res in results.items():
    resource_rows.append({
        'model': key,
        'fit_seconds': round(res['fit_seconds'], 2),
        'score_seconds': round(res['score_seconds'], 3),
        'process_rss_mb': round(rss_mb(), 1),
    })
resource_table = pd.DataFrame(resource_rows).set_index('model')
print('Logical CPUs:', psutil.cpu_count(logical=True))
print('Total RAM MB:', round(psutil.virtual_memory().total / 1024 ** 2))
resource_table

Logical CPUs: 20
Total RAM MB: 16088


,fit_seconds,score_seconds,process_rss_mb
model,,,
popularity,0.78,0.866,239.5
bias,4.03,0.224,239.6
item_cf,85.86,1.692,239.6
matrix_factorization,660.61,1.826,239.6


## 9. P2.5 — Tuning budgets

Selection uses 2017 validation only: highest mean NDCG@10, ties broken by coverage. No test data is used. The search combines the base grid (`src/tune_models.py`) with targeted extensions (`src/tune_extension.py`); budgets ended up unequal across models:

- Bias (7 settings): $\lambda_u = \lambda_i \in \{5, 10, 20, 40, 80, 160, 320\}$ → selected 320
- Item-CF (9 settings): $K \in \{3, 4, 6, 8, 12, 25, 50, 100, 200\}$ → selected 3
- MF (8 settings): $(k, \lambda) \in \{(8, 15), (8, 30), (16, 15), (16, 30), (16, 45), (24, 10), (24, 15), (24, 30)\}$ → selected (24, 30)

Bias $\lambda = 320$ and Item-CF $K = 3$ sit at the edges of their grids, as do MF's 24 factors, so better settings may lie outside the explored ranges.

The cell below shows the recorded grid. Refitting it live is optional because it refits 24 models.

In [10]:
from src.tune_models import MODEL_FACTORIES

GRIDS = {
    'bias': [
        {'lambda_user': v, 'lambda_movie': v}
        for v in (5.0, 10.0, 20.0, 40.0, 80.0, 160.0, 320.0)
    ],
    'item_cf': [
        {'top_k': k, 'min_similarity': 0.0}
        for k in (3, 4, 6, 8, 12, 25, 50, 100, 200)
    ],
    'matrix_factorization': [
        {'n_factors': k, 'regularization': r}
        for k, r in ((8, 15.0), (8, 30.0), (16, 15.0), (16, 30.0),
                     (16, 45.0), (24, 10.0), (24, 15.0), (24, 30.0))
    ],
}

RUN_LIVE_GRID = False  # set True to refit every grid entry (slow)
if RUN_LIVE_GRID:
    grid_rows = []
    for mname, grid in GRIDS.items():
        for params in grid:
            res = evaluate_model(MODEL_FACTORIES[mname](**params), cohort)
            s = res['summary']
            grid_rows.append({
                'model': mname,
                'params': str(params),
                'mean_ndcg_at_10': s['mean_ndcg_at_10'],
                'coverage_movies': s['coverage_movies'],
            })
    grid_table = pd.DataFrame(grid_rows)
    print('Live grid complete.')
else:
    grid_table = pd.read_csv('outputs/phase2_tuning_grid_extended.csv')[
        ['model', 'params', 'mean_ndcg_at_10', 'coverage_movies']
    ]
    print('Showing the recorded grid (outputs/phase2_tuning_grid_extended.csv).')
    print('Set RUN_LIVE_GRID = True to refit it here.')

grid_table.sort_values(['model', 'mean_ndcg_at_10'], ascending=[True, False])

Showing the recorded grid (outputs/phase2_tuning_grid_extended.csv).
Set RUN_LIVE_GRID = True to refit it here.


,model,params,mean_ndcg_at_10,coverage_movies
19,bias,"{""lambda_movie"": 320.0, ""lambda_user"": 320.0}",0.052504,154
4,bias,"{""lambda_movie"": 80.0, ""lambda_user"": 80.0}",0.049123,119
18,bias,"{""lambda_movie"": 160.0, ""lambda_user"": 160.0}",0.048267,137
3,bias,"{""lambda_movie"": 40.0, ""lambda_user"": 40.0}",0.043384,97
2,bias,"{""lambda_movie"": 20.0, ""lambda_user"": 20.0}",0.038258,81
0,bias,"{""lambda_movie"": 5.0, ""lambda_user"": 5.0}",0.033641,37
1,bias,"{""lambda_movie"": 10.0, ""lambda_user"": 10.0}",0.033420,59
22,item_cf,"{""min_similarity"": 0.0, ""top_k"": 3}",0.059745,4627
23,item_cf,"{""min_similarity"": 0.0, ""top_k"": 4}",0.055505,4943
20,item_cf,"{""min_similarity"": 0.0, ""top_k"": 6}",0.052409,5371


## 10. P2.9 — Self-checks and reproduction

Deterministic synthetic tests validate every model and the exclusion rule without the full archive (`src/check_phase2.py`):

- popularity ranks and refuses star predictions,
- bias beats the global-mean RMSE,
- item-CF similarity on co-liked items and `NaN` for unknown users,
- MF recovers low-rank synthetic data,
- the pipeline excludes already-rated movies,
- rating clipping: ranking scores stay unclipped, while `predict` and fallback-filled rating predictions are clipped to [0.5, 5].

In [11]:
from src import check_phase2

check_phase2.main()

popularity pass
bias pass
item_cf pass
matrix_factorization pass
pipeline exclusion pass
rating clipping pass
P2 self-checks passed


## 11. Results summary

Combine the ranking and rating tables, then compare them with the saved full validation run (`outputs/phase2_validation_summary.csv`). Reference values from that run (22 November 2026):

| Model | NDCG@10 | Recall@10 | Coverage | RMSE | MAE |
|---|---:|---:|---:|---:|---:|
| Popularity | 0.075212 | 0.023637 | 287 | — | — |
| Bias (320/320) | 0.052504 | 0.018621 | 154 | 0.8769 | 0.6596 |
| Item-CF (K=3) | 0.059745 | 0.025483 | 4,627 | 0.8796 | 0.6497 |
| MF (24, 30) | **0.095484** | **0.040890** | 1,182 | **0.7991** | **0.5902** |

In [12]:
summary_table = ranking_table.copy()
if len(rating_table):
    summary_table = summary_table.join(rating_table, how='left')

# Compare with the saved runner output: differences should be ~0.
saved = pd.read_csv('outputs/phase2_validation_summary.csv').set_index('model')
compare = ['mean_ndcg_at_10', 'mean_recall_at_10', 'coverage_movies',
           'less_popular_exposure', 'missing_scores', 'rmse', 'mae']
difference = (summary_table[compare] - saved.loc[summary_table.index, compare]).abs()
print('Largest absolute difference from the saved run:')
print(difference.max().to_string())

summary_table.round(6)

Largest absolute difference from the saved run:
mean_ndcg_at_10          0.0
mean_recall_at_10        0.0
coverage_movies          0.0
less_popular_exposure    0.0
missing_scores           0.0
rmse                     0.0
mae                      0.0


,mean_ndcg_at_10,mean_recall_at_10,coverage_movies,coverage_fraction,less_popular_exposure,missing_scores,rmse,mae
model,,,,,,,,
popularity,0.075212,0.023637,287,0.015976,0.000000,0,NaN,NaN
bias,0.052504,0.018621,154,0.008572,0.000167,0,0.876850,0.659556
item_cf,0.059745,0.025483,4627,0.257556,0.159272,81597241,0.879636,0.649669
matrix_factorization,0.095484,0.040890,1182,0.065795,0.272625,0,0.799053,0.590185


## 12. Notes and next steps

- Full reproducible run: `python -m src.run_phase2_validation` reads `outputs/phase2_selected_config_extended.json` and writes `outputs/phase2_validation_summary.csv`, per-model user metrics, top-10 lists and `phase2_validation_environment.json` (settings, fallback, clip range, git commit).
- Tuning record: `python -m src.tune_models` writes the base grid (`outputs/phase2_tuning_grid.csv`); `python -m src.tune_extension` adds the extension probes and writes the combined grid and `outputs/phase2_selected_config_extended.json`.
- Findings: MF is best on both ranking (27.0% above popularity on NDCG@10) and rating accuracy, so it is the comparator for the Phase 3 hybrid. Bias and Item-CF both rank below popularity on NDCG@10. Bias has the better RMSE of the two but the worse NDCG@10 and MAE, which illustrates the gap between rating accuracy and ranking.
- Item-CF at $K = 3$ covers the most movies (4,627) but relies on the fallback for about 95% of scores; measuring how many top-10 slots come from direct Item-CF scores is still open.
- Remaining: save MF convergence evidence, per-model peak memory if P2.8 requires it, and P2.9 independent reproduction and sign-off.